# TALN — TP 7 : Transformers – Encodeur
**FST · 2026–2027 · Dr. Asma Mekki · IDS 4 / IGL 5**

Notebook corrigé et commenté suivant les **16 questions** du sujet. Exécuter les cellules dans l’ordre. La première utilisation de `bert-base-uncased` nécessite une connexion Internet. Les calculs sont exécutables sur CPU.

## Installation et imports
`torch` est généralement préinstallé sur Colab. Ne pas réinstaller PyTorch inutilement.

In [ ]:
%pip install -q transformers

In [ ]:
import math
import torch
from torch import nn
from transformers import AutoTokenizer, AutoConfig

torch.manual_seed(42)
print("PyTorch :", torch.__version__)

## Étape 1 — Tokenisation et embeddings (questions 1 à 3)
- **input_ids** : identifiants numériques des sous-mots, y compris les tokens spéciaux.
- **attention_mask** : 1 pour les tokens réels, 0 pour le padding. Il évite que le modèle prête attention aux tokens de remplissage.

In [ ]:
phrase = "Natural language processing is interesting."
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
encoded = tokenizer(phrase, return_tensors="pt", padding=True, truncation=True)
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]
print("Tokens :", tokenizer.convert_ids_to_tokens(input_ids[0]))
print("input_ids :", input_ids)
print("attention_mask :", attention_mask)

In [ ]:
config = AutoConfig.from_pretrained("bert-base-uncased")
print("vocab_size :", config.vocab_size)
print("hidden_size :", config.hidden_size)
print("num_attention_heads :", config.num_attention_heads)
print("max_position_embeddings :", config.max_position_embeddings)

embedding_layer = nn.Embedding(config.vocab_size, config.hidden_size)
token_embeddings = embedding_layer(input_ids)
batch_size, seq_len, hidden_dim = token_embeddings.shape
print("Embeddings :", token_embeddings.shape)
assert token_embeddings.shape == (batch_size, seq_len, hidden_dim)

## Étape 2 — Encodage positionnel (questions 4 à 7)
On construit un encodage sinusoïdal :

\[PE(pos,2i)=\sin(pos/10000^{2i/d}),\qquad PE(pos,2i+1)=\cos(pos/10000^{2i/d})\]

Le facteur de division équivalent est `exp(-log(10000) * i / embed_dim)` pour les indices pairs `i`.

In [ ]:
class PositionalEmbedding(nn.Module):
    def __init__(self, embed_dim: int, max_len: int = 512):
        super().__init__()
        positions = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        even_dims = torch.arange(0, embed_dim, 2, dtype=torch.float32)
        div_term = torch.exp(-math.log(10000.0) * even_dims / embed_dim)
        pe = torch.zeros(max_len, embed_dim)
        pe[:, 0::2] = torch.sin(positions * div_term)
        # Gère également une dimension impaire.
        pe[:, 1::2] = torch.cos(positions * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        if x.size(1) > self.pe.size(1):
            raise ValueError("Séquence plus longue que max_len")
        return x + self.pe[:, :x.size(1), :].to(dtype=x.dtype)

positional_layer = PositionalEmbedding(hidden_dim, max_len=config.max_position_embeddings)
embeddings_with_position = positional_layer(token_embeddings)
print("Avec position :", embeddings_with_position.shape)
print("Premières valeurs de PE(position 0) :", positional_layer.pe[0, 0, :8])

## Étape 3 — Self-attention et multi-têtes (questions 8 à 12)
**Self-attention :** chaque token compare sa requête `Q` aux clés `K` de tous les tokens, puis forme une somme pondérée des valeurs `V`. Les poids sont obtenus par softmax des scores `QKᵀ / √d_k`.

**Pourquoi plusieurs têtes ?** Elles apprennent différentes relations entre les tokens dans plusieurs sous-espaces de représentation. Leurs sorties sont concaténées et projetées.

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """query, key, value : (batch, seq_len, dim_k) ; retourne (sortie, poids)."""
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / math.sqrt(dim_k)
    if mask is not None:
        # mask : (batch, seq_len), 1=token réel, 0=padding
        scores = scores.masked_fill(mask[:, None, :] == 0, torch.finfo(scores.dtype).min)
    weights = torch.softmax(scores, dim=-1)
    result = torch.bmm(weights, value)
    return result, weights

attention_output, attention_weights = scaled_dot_product_attention(
    embeddings_with_position, embeddings_with_position, embeddings_with_position, attention_mask
)
print("Self-attention :", attention_output.shape)
print("Poids :", attention_weights.shape)
print("Somme des poids par requête :", attention_weights.sum(dim=-1))
assert attention_output.shape == (batch_size, seq_len, hidden_dim)
assert torch.allclose(attention_weights.sum(-1), torch.ones_like(attention_weights.sum(-1)), atol=1e-5)

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim)
        self.key = nn.Linear(embed_dim, head_dim)
        self.value = nn.Linear(embed_dim, head_dim)

    def forward(self, hidden_state, mask=None):
        q = self.query(hidden_state)
        k = self.key(hidden_state)
        v = self.value(hidden_state)
        output, _ = scaled_dot_product_attention(q, k, v, mask)
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError("embed_dim doit être divisible par num_heads")
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim, head_dim) for _ in range(num_heads)])
        self.output_projection = nn.Linear(embed_dim, embed_dim)

    def forward(self, hidden_state, mask=None):
        outputs = [head(hidden_state, mask) for head in self.heads]
        return self.output_projection(torch.cat(outputs, dim=-1))

multihead = MultiHeadAttention(hidden_dim, config.num_attention_heads)
multihead_output = multihead(embeddings_with_position, attention_mask)
print("Multi-head attention :", multihead_output.shape)
assert multihead_output.shape == token_embeddings.shape

## Étape 4 — Normalisation de couche (questions 13 et 14)
On normalise sur la dernière dimension : `gamma * (x - moyenne) / sqrt(variance + eps) + beta`. Les paramètres `gamma` et `beta` sont entraînables. La variance utilise `unbiased=False` (variance de population).

In [ ]:
class CustomLayerNorm(nn.Module):
    def __init__(self, embed_dim, eps=1e-12):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(variance + self.eps) + self.beta

layer_norm = CustomLayerNorm(hidden_dim)
normalized = layer_norm(multihead_output)
print("Sortie normalisée :", normalized.shape)
print("Moyenne (premier token) :", normalized[0, 0].mean().item())
print("Variance (premier token) :", normalized[0, 0].var(unbiased=False).item())
assert torch.allclose(normalized.mean(-1), torch.zeros_like(normalized.mean(-1)), atol=1e-4)
assert torch.allclose(normalized.var(-1, unbiased=False), torch.ones_like(normalized.mean(-1)), atol=1e-3)

## Étape 5 — Feed-forward (questions 15 et 16)
Deux projections linéaires encadrent une activation GELU, suivie d’un dropout. Le sujet demande une première couche qui **réduit** la dimension puis une seconde qui la rétablit : on suit ici exactement cette consigne. Dans les Transformers classiques, la dimension intermédiaire est souvent au contraire plus grande.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, intermediate_dim=None, dropout=0.1):
        super().__init__()
        intermediate_dim = intermediate_dim or embed_dim // 2
        self.linear_1 = nn.Linear(embed_dim, intermediate_dim)
        self.activation = nn.GELU()
        self.linear_2 = nn.Linear(intermediate_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.linear_2(self.activation(self.linear_1(x))))

feed_forward = FeedForward(hidden_dim)
feed_forward.eval()  # dropout désactivé pour un test reproductible
with torch.no_grad():
    ff_output = feed_forward(multihead_output)
print("Entrée :", token_embeddings.shape)
print("Sortie feed-forward :", ff_output.shape)
assert ff_output.shape == token_embeddings.shape

## Vérification récapitulative
Ce TP implémente les composants demandés d’un encodeur Transformer. **Il ne charge pas les poids pré-entraînés de BERT** : `AutoConfig` fournit sa configuration, tandis que les couches PyTorch créées ici sont initialisées aléatoirement. Un bloc encodeur complet peut également inclure des connexions résiduelles autour de l’attention et du feed-forward.

In [ ]:
print("Toutes les vérifications de dimensions sont réussies.")
print(f"Batch={batch_size}, longueur={seq_len}, dimension={hidden_dim}, têtes={config.num_attention_heads}")